# 04 · Collect healthy AT2 cells for consistent NMF

Fourth step of the healthy-lung pipeline (paper Methods: *Non-malignant lung single cell data processing*; Figure 1B).

**Input**: `intermediate_files/lung/adata_preprocessed.h5ad` (File 02: healthy lung cells, normalised and log-transformed)

**Steps**
1. Select author-annotated AT2 cells (`ann_level_3 == "AT2"`) and remove samples with < 50 AT2 cells: 115,666 cells × 43,496 genes remain.
2. Plot AT2 cell counts by lung condition and tissue (Figure 1B; saved as `cell_counts_by_condition_and_tissue.svg`).
3. Export the AT2 expression matrix with `obs`/`var` tables for the R-based NMF in File 05.

**Output**
- `intermediate_files/lung/02_AT2_normal.h5ad`: the log-normalised expression matrix (`X`) only; the raw-count layer, `obs`, `var` and `uns` are removed to simplify the conversion to Seurat. File 05 loads this matrix as the Seurat `data` slot and runs NMF on it per sample.
- `intermediate_files/lung/02_at2_obs.csv`, `intermediate_files/lung/02_at2_var.csv`: cell and gene metadata, added back in File 05.

## Setup

In [1]:
import gc
import re

import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns

In [2]:
# Paths: edit to match your environment
DATA_DIR = "intermediate_files/lung"
IN_H5AD = f"{DATA_DIR}/adata_preprocessed.h5ad"  # output of File 02

## 1. Load data and select AT2 cells

In [3]:
adata = sc.read(IN_H5AD, citeseq=None)

In [4]:
adata.obs['lung_condition']

CGATGTAAGTTACGGG_SC10                                           Healthy
cc05p_CATGCCTGTGTGCCTG_carraro_csmc                             Healthy
ATTCTACCAAGGTTCT_HD68                                           Healthy
D062_TGACCCTTCAAACCCA-sub_wang_sub_batch3                       Healthy
P1_2_TGCTGCTAGCTCCTCT                          Healthy (tumor adjacent)
                                                         ...           
P3_4_GCTTGAACACGACGAA                          Healthy (tumor adjacent)
TTGTGGATCGTTCCTG_5-PX5-sub_mould                                Healthy
TCAGGATCAAGACGTG_F02526                                         Healthy
CAACCTCTCATGTAGC-WSSS8015042-0_meyer_unpubl                     Healthy
022C-b_GGATGTTTCCAAGTAC_adams                                   Healthy
Name: lung_condition, Length: 1292321, dtype: category
Categories (2, object): ['Healthy', 'Healthy (tumor adjacent)']

In [5]:
gc.collect()

12

In [6]:
adata = adata[adata.obs['ann_level_3'].isin(['AT2'])]

# Remove samples with fewer than 50 AT2 cells
sample_counts = adata.obs['sample'].value_counts()
valid_samples = sample_counts[sample_counts >= 50].index
adata = adata[adata.obs['sample'].isin(valid_samples)].copy()

In [7]:
print(adata.shape)

(115666, 43496)


In [8]:
adata.obs['ann_level_3'].cat.categories

Index(['AT2'], dtype='object')

In [9]:
adata

AnnData object with n_obs × n_vars = 115666 × 43496
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', "3'_or_5'", 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'core_or_extension', 'dataset', 'fresh_or_frozen', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'original_ann_level_1', 'original_ann_level_2', 'original_ann_level_3', 'original_ann_level_4', 'original_ann_level_5', 'original_ann_nonharmonized', 'reannotation_type', 'sample', 'scanvi_label', 'sequencing_platform', 'smoking_status', 'study', 'subject_type', 'tissue_coarse_unharmonized', '

## 2. AT2 cell counts by lung condition and tissue (Figure 1B)

The figure is saved to SVG and closed, so it is not displayed in the notebook.

In [10]:
try:
    plt.figure(figsize=(10, 6))

    # Directly use adata.obs for cell-level counts
    sns.countplot(
        data=adata.obs,
        x='lung_condition',
        hue='tissue',
        palette='Set2',
        edgecolor='black'
    )

    plt.title('Cell Counts by Lung Condition and Tissue Type', fontsize=16)
    plt.xlabel('Lung Condition', fontsize=12)
    plt.ylabel('Number of Cells', fontsize=12)
    plt.xticks(rotation=45, ha='right') 
    plt.legend(title='Tissue Type')
    plt.tight_layout()
    
    file_name = 'cell_counts_by_condition_and_tissue.svg'
    plt.savefig(file_name, format='svg')
    plt.close() # Close the figure to free up memory
    
    print(f"Figure successfully saved as {file_name}")
    plt.show()
except KeyError as e:
    print(f"Error: A required column {e} was not found. Ensure 'lung_condition' and 'tissue' are in adata.obs.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Figure successfully saved as cell_counts_by_condition_and_tissue.svg


In [11]:
adata.obs['disease'].cat.categories

Index(['normal'], dtype='object')

## 3. Export the AT2 expression matrix for NMF (File 05)

In [12]:
adata.var_names.name = None
adata.var['genes'] = adata.var_names

In [13]:
# Check that no gene symbols still carry an Ensembl ID suffix
pattern = re.compile(r'.+_ENSG\d+')

# Function to count the number of gene names matching the pattern
def count_matching_genes(var_names, pattern):
    count = 0
    for name in var_names:
        if pattern.match(name):
            count += 1
    return count

# Count the number of gene names matching the pattern
matching_genes_count = count_matching_genes(adata.var_names, pattern)

print(f"Number of genes matching the pattern: {matching_genes_count}")

Number of genes matching the pattern: 0


In [14]:
gc.collect()

5079

In [15]:
adata.var = adata.var.reset_index()
adata.obs['donor_id'] = adata.obs['sample'].astype('str')
adata.var['genes'] = adata.var['genes'].astype('str')

In [16]:
adata.obs.to_csv(f'{DATA_DIR}/02_at2_obs.csv')
adata.var.to_csv(f'{DATA_DIR}/02_at2_var.csv')

In [17]:
# Keep only the log-normalised expression matrix (X) for the R conversion
del adata.obs
del adata.var
del adata.layers
del adata.uns

In [18]:
adata.write(f"{DATA_DIR}/02_AT2_normal.h5ad")